In [ ]:
%pip install pandas numpy scikit-learn joblib gradio flask

In [15]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

Libraries imported successfully!


In [16]:
df = pd.read_csv(r"C:\Users\92335\Downloads\Student_Performance.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (10000, 6)


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


In [18]:
print("Column names :")
print(df.columns.tolist())

Column names :
['Hours Studied', 'Previous Scores', 'Extracurricular Activities', 'Sleep Hours', 'Sample Question Papers Practiced', 'Performance Index']


In [21]:
print("Dataset information :")
df.info()

Dataset information :
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Hours Studied                     10000 non-null  int64  
 1   Previous Scores                   10000 non-null  int64  
 2   Extracurricular Activities        10000 non-null  str    
 3   Sleep Hours                       10000 non-null  int64  
 4   Sample Question Papers Practiced  10000 non-null  int64  
 5   Performance Index                 10000 non-null  float64
dtypes: float64(1), int64(4), str(1)
memory usage: 468.9 KB


In [22]:
print("Missing values :")
print(df.isnull().sum())

Missing values :
Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64


In [23]:
print("Duplicate records :")
print(df.duplicated().sum())

Duplicate records :
127


In [24]:
print("Statistical summary:")
display(df.describe())

Statistical summary:


,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Performance Index
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,4.992900,69.445700,6.530600,4.583300,55.224800
std,2.589309,17.343152,1.695863,2.867348,19.212558
min,1.000000,40.000000,4.000000,0.000000,10.000000
25%,3.000000,54.000000,5.000000,2.000000,40.000000
50%,5.000000,69.000000,7.000000,5.000000,55.000000
75%,7.000000,85.000000,8.000000,7.000000,71.000000
max,9.000000,99.000000,9.000000,9.000000,100.000000


In [25]:
df = df.drop_duplicates().copy()

df = df.dropna().copy()

df["Extracurricular Activities"] = (
    df["Extracurricular Activities"]
    .map({"Yes": 1, "No": 0})
)

df = df.dropna().copy()

print("Data cleaned successfully!")
print("Remaining records:", len(df))

display(df.head())

Data cleaned successfully!
Remaining records: 9873


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,1,9,1,91.0
1,4,82,0,4,2,65.0
2,8,51,1,7,2,45.0
3,5,52,1,5,2,36.0
4,7,75,0,8,5,66.0


In [27]:
features = ["Hours Studied","Previous Scores","Extracurricular Activities","Sleep Hours","Sample Question Papers Practiced"]

X = df[features]
y = df["Performance Index"]

print("Input features (X):")
display(X.head())

print("Target variable (y):")
display(y.head())

Input features (X):


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced
0,7,99,1,9,1
1,4,82,0,4,2
2,8,51,1,7,2
3,5,52,1,5,2
4,7,75,0,8,5


Target variable (y):


0    91.0
1    65.0
2    45.0
3    36.0
4    66.0
Name: Performance Index, dtype: float64

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 7898
Testing records: 1975


In [29]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Linear Regression model trained successfully!")

Linear Regression model trained successfully!


In [31]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MODEL EVALUATION")

print("\nMAE:", round(mae, 3))
print("MSE:", round(mse, 3))
print("RMSE:", round(rmse, 3))
print("R2 Score:", round(r2, 4))

MODEL EVALUATION

MAE: 1.647
MSE: 4.306
RMSE: 2.075
R2 Score: 0.9884


In [32]:
coefficients = pd.DataFrame({"Feature": features,"Coefficient": model.coef_})

display(coefficients)

print("Intercept:", model.intercept_)

,Feature,Coefficient
0,Hours Studied,2.851022
1,Previous Scores,1.018430
2,Extracurricular Activities,0.573823
3,Sleep Hours,0.472073
4,Sample Question Papers Practiced,0.188704


Intercept: -33.981324496440706


In [33]:
new_student = pd.DataFrame([{"Hours Studied": 6,"Previous Scores": 80,"Extracurricular Activities": 1,"Sleep Hours": 7,"Sample Question Papers Practiced": 4}])

predicted_score = model.predict(new_student)[0]

print("Predicted Performance Index:", round(predicted_score, 2))

Predicted Performance Index: 69.23


In [34]:
joblib.dump({"model": model,"features": features},"student_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [35]:
import gradio as gr

def predict_performance(
    hours_studied,
    previous_scores,
    extracurricular,
    sleep_hours,
    practice_papers
):
    student = pd.DataFrame([{
        "Hours Studied": hours_studied,
        "Previous Scores": previous_scores,
        "Extracurricular Activities": (
            1 if extracurricular == "Yes" else 0
        ),
        "Sleep Hours": sleep_hours,
        "Sample Question Papers Practiced": practice_papers
    }], columns=features)

    prediction = model.predict(student)[0]

    return round(float(prediction), 2)


demo = gr.Interface(
    fn=predict_performance,
    inputs=[
        gr.Number(label="Hours Studied", value=5),
        gr.Number(label="Previous Scores", value=75),
        gr.Radio(
            choices=["Yes", "No"],
            label="Extracurricular Activities",
            value="Yes"
        ),
        gr.Number(label="Sleep Hours", value=7),
        gr.Number(
            label="Sample Question Papers Practiced",
            value=4
        )
    ],
    outputs=gr.Number(label="Predicted Performance Index"),
    title="Student Performance Prediction",
    description="Enter student details to predict the Performance Index."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [36]:
from pathlib import Path

Path("templates").mkdir(exist_ok=True)

print("Templates folder created!")

Templates folder created!


In [37]:
html_code = """
<!DOCTYPE html>
<html>
<head>
    <title>Student Performance Prediction</title>
    <style>
        body {
            font-family: Arial;
            background: #f2f4f8;
            padding: 30px;
        }
        .container {
            max-width: 500px;
            margin: auto;
            background: white;
            padding: 25px;
            border-radius: 12px;
        }
        label {
            display: block;
            margin-top: 15px;
        }
        input, select, button {
            width: 100%;
            padding: 10px;
            margin-top: 6px;
            box-sizing: border-box;
        }
        button {
            margin-top: 20px;
            background: #2457c5;
            color: white;
            border: none;
            cursor: pointer;
        }
        .result {
            margin-top: 20px;
            background: #e7f7e9;
            padding: 12px;
            text-align: center;
        }
    </style>
</head>
<body>
    <div class="container">
        <h2>Student Performance Prediction</h2>

        <form method="POST">
            <label>Hours Studied</label>
            <input type="number" name="hours_studied"
                   min="0" step="any" required>

            <label>Previous Scores</label>
            <input type="number" name="previous_scores"
                   min="0" step="any" required>

            <label>Extracurricular Activities</label>
            <select name="extracurricular">
                <option value="Yes">Yes</option>
                <option value="No">No</option>
            </select>

            <label>Sleep Hours</label>
            <input type="number" name="sleep_hours"
                   min="0" step="any" required>

            <label>Sample Question Papers Practiced</label>
            <input type="number" name="practice_papers"
                   min="0" step="1" required>

            <button type="submit">Predict Performance</button>
        </form>

        {% if prediction is not none %}
        <div class="result">
            <h3>Predicted Performance Index</h3>
            <h2>{{ prediction }}</h2>
        </div>
        {% endif %}

        {% if error %}
        <p>{{ error }}</p>
        {% endif %}
    </div>
</body>
</html>
"""

Path("templates/index.html").write_text(html_code, encoding="utf-8")

print("HTML template created successfully!")

HTML template created successfully!


In [38]:
from flask import Flask, render_template, request

app = Flask("student_performance_app")

@app.route("/", methods=["GET", "POST"])
def home():
    prediction = None
    error = None

    if request.method == "POST":
        try:
            hours_studied = float(request.form["hours_studied"])
            previous_scores = float(request.form["previous_scores"])
            extracurricular = request.form["extracurricular"]
            sleep_hours = float(request.form["sleep_hours"])
            practice_papers = float(request.form["practice_papers"])

            if min(
                hours_studied,
                previous_scores,
                sleep_hours,
                practice_papers
            ) < 0:
                raise ValueError

            if extracurricular not in ["Yes", "No"]:
                raise ValueError

            student = pd.DataFrame([{
                "Hours Studied": hours_studied,
                "Previous Scores": previous_scores,
                "Extracurricular Activities": (
                    1 if extracurricular == "Yes" else 0
                ),
                "Sleep Hours": sleep_hours,
                "Sample Question Papers Practiced": practice_papers
            }], columns=features)

            prediction = round(
                float(model.predict(student)[0]), 2
            )

        except (ValueError, KeyError):
            error = "Please enter valid, non-negative values."

    return render_template(
        "index.html",
        prediction=prediction,
        error=error
    )

print("Flask app is ready!")

Flask app is ready!


In [ ]:
app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)

 * Serving Flask app 'student_performance_app'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/Oct/2026 23:48:32] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Oct/2026 23:48:33] "GET /favicon.ico HTTP/1.1" 404 -
